**Purpose:** Run the complete 5-condition experiment: C1 → C2 → ChromaDB → DSPy → C3 (RAG+DSPy) → C4 (Feedback) → C5 (Runtime Validation).

**Runtime:** Colab A100 High-RAM
**Model:** Qwen 2.5-coder:32b via Ollama

---
### Pipeline overview
| # | Phase | Script | Est. time |
|---|-------|--------|-----------|
| 1–5 | Setup | Drive, deps, Ollama, repo | 10 min |
| 6 | C1 Baseline | `multi_main.py` | ~4 hrs (500 signals) |
| 7 | C2 Adaptive | `multi_main_adaptive.py` | ~4 hrs (500 signals) |
| 8 | ChromaDB | `rag.aosp_indexer` | 5 min (or 30s from cache) |
| 9 | DSPy Optimiser | `dspy_opt/optimizer.py` | 2–3 hrs (train-size 8) |
| 10 | Restore + Preflight | — | 1 min |
| 11 | Bug fixes | — | instant |
| 12 | C3 RAG+DSPy | `multi_main_rag_dspy.py` | ~4 hrs (500 signals) |
| 13 | C4 Feedback | `multi_main_c4_feedback.py` | ~4 hrs (500 signals) |
| 14 | Reporting | diagnose → rescore → compare | 2 min |
| 15 | Export | — | 1 min |
| 16 | C5 Runtime Validation | `multi_main_c5.py` | ~1 hr |
| 17 | Utilities | — | as needed |

### Changes in v10 (vs v9)
- **C5 pipeline added** — advanced runtime validation (FakeVHAL patch + VTS + HMI app)
- **500 signals** — `TEST_SIGNAL_COUNT = 500`, `random.seed(42)`
- **Domain-specific AIDL base addresses** — ADAS=0x1000, Body=0x2000, Cabin=0x3000, etc.
- **`FUTURE_TIMEOUT = 21600`** in `multi_main.py` — prevents timeout on 7 modules
- **App agent optimized** — `MAX_PROPS_PER_CHUNK = 40`, reduced layout timeouts
- **Statistical significance confirmed** — Kruskal-Wallis p=0.040, C1 vs C4 p=0.016

## 1. Configuration

In [1]:
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [22]:
# ── Configuration — edit these paths once ─────────────────────────
import os, time, shutil, glob, subprocess, requests
from pathlib import Path
import chromadb

DRIVE_SRC    = "/content/drive/MyDrive/mse25_thesis"
REPO_URL     = "https://github.com/appdev1307/code-codegen-aosp-llm-based.git"
REPO_DIR     = "/content/code-codegen-aosp-llm-based"
MODEL_NAME   = "qwen2.5-coder:32b"
OLLAMA_LOG   = "/content/ollama.log"

# AOSP source & ChromaDB
AOSP_SRC_DIR = f"{REPO_DIR}/aosp_source"
CHROMA_DB    = f"{REPO_DIR}/rag/chroma_db"
CHROMA_ZIP   = f"{DRIVE_SRC}/chroma_db.zip"

# AOSP version — must match the build tree on GCP
AOSP_TAG     = "android-14.0.0_r75"

# Restore mappings
RESTORE_MAP = {
    "output_c1.zip":          f"{REPO_DIR}/output",
    "output_c2.zip":          f"{REPO_DIR}/output_adaptive",
    "output_c3.zip":          f"{REPO_DIR}/output_rag_dspy",
    "output_c4.zip":          f"{REPO_DIR}/output_c4_feedback",
    "output_c5.zip":          f"{REPO_DIR}/output_c5",
    "dspy_saved_backup.zip":  f"{REPO_DIR}/dspy_opt/saved",
}

# Ollama parallelism — uses more VRAM but speeds up DSPy 2-3x
os.environ["OLLAMA_NUM_PARALLEL"] = "4"

print("✓ Config loaded (OLLAMA_NUM_PARALLEL=4, AOSP_TAG=" + AOSP_TAG + ")")

✓ Config loaded (OLLAMA_NUM_PARALLEL=4, AOSP_TAG=android-14.0.0_r75)


## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_SRC).mkdir(parents=True, exist_ok=True)
print(f"✓ Drive mounted — {DRIVE_SRC}")

Mounted at /content/drive
✓ Drive mounted — /content/drive/MyDrive/mse25_thesis


## 3. System dependencies

In [4]:
%%capture
!apt-get update -qq
!apt-get install -y -qq clang checkpolicy zstd

for tool in ["clang", "checkpolicy", "zstd"]:
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    print(f"  {tool}: {'✓ ' + path if path else '❌ NOT FOUND'}")

## 4. Ollama setup

In [5]:
def start_ollama():
    """Install (if needed), start Ollama with parallel support, wait until healthy."""
    if not Path("/usr/local/bin/ollama").exists():
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True)
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(1)
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT,
                     env={**os.environ})
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"✓ Ollama server ready (took {i+1}s, parallel={os.environ.get('OLLAMA_NUM_PARALLEL', '1')})")
                return True
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("❌ Ollama failed to start — check ollama.log")

start_ollama()

Installing Ollama...
✓ Ollama server ready (took 2s, parallel=4)


True

In [6]:
# Pull model (skip if already cached)
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL_NAME.split(":")[0] in result.stdout:
    print(f"✓ Model {MODEL_NAME} already available")
else:
    print(f"Pulling {MODEL_NAME}...")
    !ollama pull {MODEL_NAME}
!ollama list

Pulling qwen2.5-coder:32b...

NAME                 ID              SIZE     MODIFIED               
qwen2.5-coder:32b    b92d6a0bd47e    19 GB    Less than a second ago    


## 5. Clone & setup repository

In [7]:
if Path(REPO_DIR).exists():
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "stash"], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR)
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f"✓ Working directory: {os.getcwd()}")

Cloning into '/content/code-codegen-aosp-llm-based'...
remote: Enumerating objects: 1400, done.
remote: Total 1400 (delta 0), reused 0 (delta 0), pack-reused 1400 (from 1)
Receiving objects: 100% (1400/1400), 724.68 KiB | 17.25 MiB/s, done.
Resolving deltas: 100% (909/909), done.
/content/code-codegen-aosp-llm-based
✓ Working directory: /content/code-codegen-aosp-llm-based


In [8]:
%%capture
!pip install -q -r requirements.txt
!pip install -q requests pyyaml chromadb sentence-transformers dspy-ai
!pip install -q jinja2 fastapi uvicorn pydantic
!pip install -q optuna
print("✓ Dependencies installed (including optuna for MIPROv2)")

## 6. Run C1 — Baseline
> Vanilla LLM generation with AIDL-only prompts (~30 min).
> Agent prompts now enforce Android 14 AIDL patterns — no HIDL generation.

In [ ]:
start_ollama()
!rm -rf {REPO_DIR}/output
!python multi_main.py

✓ Ollama server ready (took 2s, parallel=4)
  VSS → AAOS HAL Generation Pipeline (LLM-First Mode)
Test signals: 500
Persistent cache: /content/vss_temp
Project output: /content/code-codegen-aosp-llm-based/output

LLM-First Configuration:
  - Goal: 90%+ LLM-generated production code
  - Strategy: Generous timeouts + progressive generation
  - Expected: Higher quality, longer generation time
  - Agents: design_doc, android_app, backend (all LLM-First)

[PREP] Loading and flattening ./dataset/vss.json ...
 Flattened to 1571 leaf signals
Selected 500 leaf signals for labelling & processing
 Wrote selected flat subset → /content/vss_temp/VSS_LIMITED_500.json
[LABELLING] Labelling the selected subset (fast mode)...
[LABELLING] Labelling 500 pre-selected signals (sequential batches)...
Labelling signals:   0%|                                                | 0/500 [00:00<?, ?signal/s][LABELLING] Batch 1/125 (4 signals)...
[INFO] Length mismatch (1 vs 4) — padding
Labelling signals:   0%|     

In [ ]:
# Verify C1 generates AIDL (not HIDL) includes
print("=== C1 C++ includes ===")
!head -10 output/hardware/interfaces/automotive/vehicle/impl/*.cpp 2>/dev/null || echo "No C++ file found"
print("\n=== C1 AIDL files ===")
!find output -name "*.aidl" -not -path "*/.llm_draft/*" | head -5

=== C1 C++ includes ===
#include <aidl/android/hardware/automotive/vehicle/BnIVehicle.h>
#include <aidl/android/hardware/automotive/vehicle/IVehicleCallback.h>
#include <aidl/android/hardware/automotive/vehicle/VehiclePropValue.h>
#include <aidl/android/hardware/automotive/vehicle/VehiclePropConfig.h>
#include <aidl/android/hardware/automotive/vehicle/VehicleArea.h>
#include <aidl/android/hardware/automotive/vehicle/VehiclePropertyAccess.h>
#include <aidl/android/hardware/automotive/vehicle/VehiclePropertyChangeMode.h>
#include "VssPropertyIds.h"

using namespace aidl::android::hardware::automotive::vehicle;

=== C1 AIDL files ===
output/hardware/interfaces/automotive/vehicle/aidl/android/hardware/automotive/vehicle/VehiclePropertyAdas.aidl


In [ ]:
# Check if scoring ran
!ls output/SCORES*.json 2>/dev/null

# If no scores file, run the scorer manually
!python experiments/analyze_results.py

  analyze_results.py
Traceback (most recent call last):
  File "/content/code-codegen-aosp-llm-based/experiments/analyze_results.py", line 326, in <module>
    main()
  File "/content/code-codegen-aosp-llm-based/experiments/analyze_results.py", line 288, in main
    data = load_comparison()
           ^^^^^^^^^^^^^^^^^
  File "/content/code-codegen-aosp-llm-based/experiments/analyze_results.py", line 49, in load_comparison
    raise FileNotFoundError(
FileNotFoundError: experiments/results/comparison.json not found.
Run: python experiments/run_comparison.py


In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/output_c1", "zip", f"{REPO_DIR}/output")
print(f"✓ C1 saved to Drive ({len(list(Path(f'{REPO_DIR}/output').rglob('*')))} files)")

✓ C1 saved to Drive (188 files)


## 7. Run C2 — Adaptive
> Adaptive generation with Thompson Sampling (~30 min).

In [20]:
!rm -rf {REPO_DIR}/output_adaptive

In [16]:
start_ollama()
!rm -rf {REPO_DIR}/output_adaptive
!python multi_main_adaptive.py

✓ Ollama server ready (took 2s, parallel=4)
[ADAPTIVE] Detected LLM model from llm_client.py: qwen2.5-coder:32b
  VSS → AAOS HAL Generation Pipeline (LLM-First + Adaptive Mode)
Test signals: 500
Persistent cache: /content/vss_temp
Project output: /content/code-codegen-aosp-llm-based/output_adaptive

LLM-First Configuration:
  - Goal: 90%+ LLM-generated production code
  - Strategy: Generous timeouts + progressive generation
  - Expected: Higher quality, longer generation time
  - Agents: design_doc, android_app, backend (all LLM-First)

[ADAPTIVE] Initializing adaptive learning components...
✓ Performance tracker database initialized: adaptive_outputs/performance_history.db
✓ Thompson Sampling Optimizer initialized
  Chunk size options: [10, 15, 20, 25, 30]
✓ Adaptive Prompt Selector initialized
  Available variants: ['minimal', 'detailed', 'conservative', 'aggressive']
✓ Adaptive Generation Wrapper initialized
  LLM Model: qwen2.5-coder:32b
  Adaptive chunking: True
  Adaptive prompts

In [ ]:
shutil.make_archive(f"{DRIVE_SRC}/output_c2", "zip", f"{REPO_DIR}/output_adaptive")
print(f"✓ C2 saved to Drive ({len(list(Path(f'{REPO_DIR}/output_adaptive').rglob('*')))} files)")

✓ C2 saved to Drive (166 files)


## 8. Build ChromaDB from AOSP source (or restore from cache)
> **Must run before DSPy.** Pinned to `android-14.0.0_r75` to match build tree.
> HIDL files excluded — AIDL-only corpus.

**Important:** If you previously cached ChromaDB without HIDL filtering or version pinning,
delete the cache first: `!rm -f {DRIVE_SRC}/chroma_db.zip`

In [9]:
# ── 8a. Try restoring from Drive ──────────────────────────────────
chroma_restored = False
if Path(CHROMA_ZIP).exists():
    print(f"Found cached ChromaDB: {CHROMA_ZIP}")
    target = Path(CHROMA_DB)
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(CHROMA_ZIP, CHROMA_DB)
    chroma_restored = True
    print("✓ ChromaDB restored from Drive cache")
else:
    print("⚠ No cached ChromaDB — will build from AOSP source")

⚠ No cached ChromaDB — will build from AOSP source


In [10]:
# ── 8b. Clone AOSP pinned to android-14.0.0_r75 (only if not restored) ─
if not chroma_restored:
    print(f"Cloning AOSP source (shallow, pinned to {AOSP_TAG}, ~300 MB)...\n")
    aosp_repos = [
        ("https://android.googlesource.com/platform/hardware/interfaces", "hardware"),
        ("https://android.googlesource.com/platform/system/sepolicy",     "sepolicy"),
        ("https://android.googlesource.com/platform/packages/services/Car", "car"),
    ]
    Path(AOSP_SRC_DIR).mkdir(parents=True, exist_ok=True)
    for url, name in aosp_repos:
        dest = f"{AOSP_SRC_DIR}/{name}"
        if Path(dest).exists():
            print(f"  ✓ {name} already cloned")
            continue
        print(f"  Cloning {name}...")
        r = subprocess.run(
            ["git", "clone", "--depth=1", "-b", AOSP_TAG, url, dest],
            capture_output=True, text=True
        )
        print(f"  {'✓' if r.returncode == 0 else '❌'} {name}")
    print(f"\n✓ AOSP source ready ({AOSP_TAG})")
else:
    print("Skipping — ChromaDB restored from cache")

Cloning AOSP source (shallow, pinned to android-14.0.0_r75, ~300 MB)...

  Cloning hardware...
  ✓ hardware
  Cloning sepolicy...
  ✓ sepolicy
  Cloning car...
  ✓ car

✓ AOSP source ready (android-14.0.0_r75)


In [11]:
# ── 8c. Index AOSP → ChromaDB (AIDL-only, HIDL excluded) ────────
if not chroma_restored:
    print("Running AOSP indexer (AIDL-only filter)...\n")
    !python -m rag.aosp_indexer --source {AOSP_SRC_DIR} --db {CHROMA_DB} --force
    print("\n✓ Indexing complete")
else:
    print("Skipping — ChromaDB already populated")

Running AOSP indexer (AIDL-only filter)...

<frozen runpy>:128: RuntimeWarning: 'rag.aosp_indexer' found in sys.modules after import of package 'rag', but prior to execution of 'rag.aosp_indexer'; this may result in unpredictable behaviour
INFO: No device provided, using cuda:0
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
modules.json: 100% 349/349 [00:00<00:00, 1.93MB/s]
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporar

In [12]:
# ── 8d. Verify ChromaDB ──────────────────────────────────────────
client = chromadb.PersistentClient(path=CHROMA_DB)
collections = client.list_collections()
total_chunks = sum(col.count() for col in collections)
print(f"  Collections: {len(collections)}, Total chunks: {total_chunks}")
for col in collections:
    print(f"    → {col.name}: {col.count()}")
assert total_chunks > 0, "❌ ChromaDB is EMPTY!"
print(f"\n✓ ChromaDB verified")
del client

  Collections: 7, Total chunks: 17643
    → aosp_car_api: 3009
    → aosp_build: 455
    → aosp_docs: 322
    → aosp_selinux: 3678
    → aosp_cpp: 1475
    → aosp_aidl: 8524
    → aosp_vintf: 180

✓ ChromaDB verified


In [13]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DB)

collections = client.list_collections()
total_chunks = sum(col.count() for col in collections)

print(f"✅ ChromaDB successfully loaded")
print(f"Collections: {len(collections)} | Total chunks: {total_chunks}\n")

for col in collections:
    print(f" → {col.name}: {col.count()} documents")

✅ ChromaDB successfully loaded
Collections: 7 | Total chunks: 17643

 → aosp_car_api: 3009 documents
 → aosp_build: 455 documents
 → aosp_docs: 322 documents
 → aosp_selinux: 3678 documents
 → aosp_cpp: 1475 documents
 → aosp_aidl: 8524 documents
 → aosp_vintf: 180 documents


In [ ]:
# ── 8e. Save to Drive ────────────────────────────────────────────
if not chroma_restored and not Path(CHROMA_ZIP).exists():
    shutil.make_archive(CHROMA_ZIP.replace(".zip", ""), "zip", CHROMA_DB)
    print(f"✓ Saved to {CHROMA_ZIP}")
else:
    print("ChromaDB cache exists — skipping save")

## 9. Run DSPy optimiser (MIPROv2)
> Runs **after** ChromaDB so traces include RAG context.
> `train-size 8` for thesis-quality (~2-3 hrs). **Skip if programs already saved.**

In [23]:
# Check if DSPy programs already exist
dspy_count = len(glob.glob("dspy_opt/saved/*/program.json"))
if dspy_count == 12:
    print(f"✓ DSPy programs already exist: {dspy_count}/12 — SKIPPING")
    print("  Delete dspy_opt/saved/ to force re-optimization")
else:
    start_ollama()
    !python dspy_opt/optimizer.py --mipro-auto light --train-size 4 --force
    dspy_count = len(glob.glob("dspy_opt/saved/*/program.json"))
    print(f"\n✓ DSPy programs: {dspy_count}/12")

✓ Ollama server ready (took 2s, parallel=4)
14:21:33 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
14:21:33 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
[Optimizer] Connecting to LLM: ollama/qwen2.5-coder:32b @ http://localhost:11434
[Optimizer] LM configured ✓
14:21:47 INFO: [RAG Retriever] Ready — DB: rag/chroma_db
[Optimizer] RAG retriever ready ✓
14:21:47 INFO: [Optimizer] Loaded 500 labelled signals from /content/vss_temp/VSS_LABELLED_500.json

[Optimizer] Optimising 12 agents: ['aidl', 'cpp', 'selinux', 'build', 'vintf', 'design_doc', 'puml', 'android_app', 'android_layout', 'backend', 'backend_model', 'simulator']
  Training examples per agent : 4
  MIPROv2 setting             : light

In [24]:
shutil.make_archive(f"{DRIVE_SRC}/dspy_saved_backup", "zip", f"{REPO_DIR}/dspy_opt/saved")
print("✓ DSPy programs saved to Drive")

✓ DSPy programs saved to Drive


## 10. Restore cached outputs & preflight check
> Restores C1/C2/DSPy from Drive if not on disk (e.g. after Colab restart).

In [25]:
# ── 10a. Restore if missing ───────────────────────────────────────
for zip_name, target_dir in RESTORE_MAP.items():
    src_path = f"{DRIVE_SRC}/{zip_name}"
    target = Path(target_dir)
    if target.exists() and len(list(target.rglob("*"))) > 0:
        print(f"  ✓ {zip_name}: on disk")
        continue
    if not Path(src_path).exists():
        print(f"  ⚠ {zip_name}: not on Drive")
        continue
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(src_path, target_dir)
    print(f"  ✓ {zip_name}: restored")

print(f"\n  DSPy programs: {len(glob.glob(f'{REPO_DIR}/dspy_opt/saved/*/program.json'))}/12")

  ✓ output_c1.zip: on disk
  ✓ output_c2.zip: on disk
  ✓ dspy_saved_backup.zip: on disk

  DSPy programs: 12/12


In [26]:
# ── 10b. Preflight ───────────────────────────────────────────────
errors = []

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=2)
    models = [m["name"] for m in r.json().get("models", [])]
    if any(MODEL_NAME.split(":")[0] in m for m in models):
        print(f"  ✓ Ollama: {MODEL_NAME}")
    else:
        errors.append(f"Model {MODEL_NAME} not found")
except Exception:
    errors.append("Ollama not responding")

try:
    client = chromadb.PersistentClient(path=CHROMA_DB)
    chunks = sum(c.count() for c in client.list_collections())
    del client
    if chunks > 0:
        print(f"  ✓ ChromaDB: {chunks} chunks")
    else:
        errors.append("ChromaDB empty")
except Exception as e:
    errors.append(f"ChromaDB: {e}")

dspy_n = len(glob.glob(f"{REPO_DIR}/dspy_opt/saved/*/program.json"))
print(f"  ✓ DSPy: {dspy_n} programs")

for label, path in [
    ("C1", f"{REPO_DIR}/output"),
    ("C2", f"{REPO_DIR}/output_adaptive"),
    ("C3", f"{REPO_DIR}/output_rag_dspy"),
    ("C4", f"{REPO_DIR}/output_c4_feedback"),
]:
    if Path(path).exists() and len(list(Path(path).rglob("*"))) > 0:
        print(f"  ✓ {label}: present")
    else:
        print(f"  ⚠ {label}: missing (will be generated or restore from Drive)")

for tool in ["clang", "checkpolicy"]:
    p = subprocess.run(["which", tool], capture_output=True, text=True)
    if p.stdout.strip():
        print(f"  ✓ {tool}")
    else:
        errors.append(f"{tool} not found")

if errors:
    print(f"\n❌ PREFLIGHT FAILED:")
    for e in errors: print(f"  • {e}")
    raise RuntimeError("Fix errors above")
else:
    print(f"\n✓ Preflight passed")

  ✓ Ollama: qwen2.5-coder:32b
  ✓ ChromaDB: 17643 chunks
  ✓ DSPy: 12 programs
  ✓ C1: present
  ✓ C2: present
  ✓ clang
  ✓ checkpolicy

✓ Preflight passed


## 11. Apply scoring & output fixes
> 1. `lstrip("**/")` → `removeprefix("**/")` — glob bug
> 2. `_clean_output()` on backend sub-modules — markdown fences
> 3. `output_root` on architect agent — wrong directory

In [27]:
# Fix 1: lstrip → removeprefix
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists(): continue
    src = Path(fpath).read_text()
    if 'pattern.lstrip("**/")' in src:
        Path(fpath).write_text(src.replace('pattern.lstrip("**/")', 'pattern.removeprefix("**/")'))
        print(f"  ✓ {label}: lstrip → removeprefix")
    else:
        print(f"  ✓ {label}: already patched")

# Fix 2: _clean_output() on backend sub-modules
be_path = f"{REPO_DIR}/agents/rag_dspy_backend_agent.py"
if Path(be_path).exists():
    be = Path(be_path).read_text()
    patched = False
    for old, new, tag in [
        ('model_content = getattr(result, "models_code", "") or ""\n                self._write',
         'model_content = getattr(result, "models_code", "") or ""\n                model_content = self._clean_output(model_content)\n                self._write', "model"),
        ('sim_content = getattr(result, "simulator_code", "") or ""\n                self._write',
         'sim_content = getattr(result, "simulator_code", "") or ""\n                sim_content = self._clean_output(sim_content)\n                self._write', "simulator"),
    ]:
        if old in be and f"_clean_output({tag}_content)" not in be:
            be = be.replace(old, new); patched = True
    if patched:
        Path(be_path).write_text(be); print("  ✓ Backend: _clean_output() added")
    else:
        print("  ✓ Backend: already patched")

# Fix 3: output_root on architect agent
for label, fpath in [("C3", f"{REPO_DIR}/multi_main_rag_dspy.py"),
                     ("C4", f"{REPO_DIR}/multi_main_c4_feedback.py")]:
    if not Path(fpath).exists(): continue
    src = Path(fpath).read_text()
    old = 'agent = RAGDSPyArchitectAgent(**AGENT_CFG)'
    new = 'agent = RAGDSPyArchitectAgent(**AGENT_CFG, output_root=str(OUTPUT_DIR))'
    if old in src:
        Path(fpath).write_text(src.replace(old, new))
        print(f"  ✓ {label}: output_root added")
    else:
        print(f"  ✓ {label}: already patched")

print("\n✓ All fixes applied")

  ✓ C3: already patched
  ✓ C4: already patched
  ✓ Backend: already patched
  ✓ C3: already patched
  ✓ C4: already patched

✓ All fixes applied


## 12. Run C3 — RAG + DSPy optimised prompts
> AIDL-only RAG corpus pinned to `android-14.0.0_r75` (~20 min).
> Agent prompts enforce AIDL patterns — C++ should use `aidl::` namespace.

In [28]:
!rm -rf {REPO_DIR}/output_rag_dspy
start_ollama()
!python apply_chroma_fix.py
!python multi_main_rag_dspy.py

✓ Ollama server ready (took 2s, parallel=4)
  Backup → multi_main_rag_dspy.py.bak
✓ Patched multi_main_rag_dspy.py at line 56
  Now re-run: python multi_main_rag_dspy.py
17:48:20 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
17:48:20 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
[RAG] ChromaDB singleton initialised (PersistentClient): 322 documents in 'aosp_docs'
[RAG] ChromaDB monkey-patched — all agents will share singleton client
  VSS → AAOS HAL Generation — Condition 3: RAG + DSPy
  Signals       : 500
  Output dir    : /content/code-codegen-aosp-llm-based/output_rag_dspy
  RAG db        : rag/chroma_db
  DSPy programs : dspy_opt/saved
  Max parallel  : 1


[Validators] Tool availability

In [29]:
# Verify C3 C++ uses AIDL (not HIDL)
print("=== C3 C++ includes ===")
!head -10 output_rag_dspy/hardware/interfaces/automotive/vehicle/impl/*.cpp 2>/dev/null || echo "No C++ file"
print("\n=== C3 AIDL files ===")
!find output_rag_dspy -name "*.aidl" | head -5

=== C3 C++ includes ===
==> output_rag_dspy/hardware/interfaces/automotive/vehicle/impl/VehicleHalServiceAdas.cpp <==
#include <aidl/android/hardware/automotive/vehicle/BnVehicle.h>
#include <VehicleHalTypes.h>
#include <VehicleUtils.h>

using namespace aidl::android::hardware::automotive::vehicle;

namespace vendor {
namespace vss {
namespace adas {


==> output_rag_dspy/hardware/interfaces/automotive/vehicle/impl/VehicleHalServiceBody.cpp <==
#include <aidl/android/hardware/automotive/vehicle/BnVehicle.h>
#include <VehicleHalTypes.h>
#include <VehicleUtils.h>
#include <utils/Log.h>
#include <vector>

using namespace aidl::android::hardware::automotive::vehicle;
using namespace ndk;

namespace {

==> output_rag_dspy/hardware/interfaces/automotive/vehicle/impl/VehicleHalServiceCabin.cpp <==
// Copyright (C) 2023 The Android Open Source Project
//
// Licensed under the Apache License, Version 2.0 (the "License");
// you may not use this file except in compliance with the License.
// You

## 13. Run C4 — Feedback loop
> Generate → validate → refine, up to 3 retries (~20 min).

In [17]:
!git pull origin main

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 346 bytes | 173.00 KiB/s, done.
From https://github.com/appdev1307/code-codegen-aosp-llm-based
 * branch            main       -> FETCH_HEAD
   510fd4c..14df3dc  main       -> origin/main
Updating 510fd4c..14df3dc
Fast-forward
 dspy_opt/optimizer.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [31]:
!rm -rf {REPO_DIR}/output_c4_feedback
start_ollama()
!python multi_main_c4_feedback.py

✓ Ollama server ready (took 2s, parallel=4)
[RAG] ChromaDB singleton initialised (PersistentClient): 322 documents in 'aosp_docs'
[RAG] ChromaDB monkey-patched — all agents will share singleton client
19:14:07 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
19:14:07 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
  VSS → AAOS HAL Generation — Condition 4: Feedback Loop
  Signals       : 500
  Output dir    : /content/code-codegen-aosp-llm-based/output_c4_feedback
  Max retries   : 3
  RAG top-k     : 8
  Strategy      : C3 architect.run() + post-validation retry


[Validators] Tool availability report:
  Agent              Tool                             Status     Note
  ───────────────────────

## 14. Reporting & analysis

In [32]:
!python diagnose_outputs.py
!python rescore_all_conditions.py
!python compare_matched.py
!python analyze_final.py

OUTPUT DIRECTORY DIAGNOSTIC
Working directory: /content/code-codegen-aosp-llm-based

── Checking known output directories ──
  ✓ output/  exists  (132 files)
  ✓ output_adaptive/  exists  (114 files)
  ✓ output_rag_dspy/  exists  (63 files)
  ✗ outputs/  not found
  ✗ output/results/  not found
  ✗ results/  not found
  ✗ generated/  not found

── output/ ──
  Total files: 132
  Extensions: {'(no ext)': 1, '.aidl': 2, '.bp': 1, '.cpp': 2, '.h': 2, '.java': 7, '.json': 3, '.kt': 27, '.md': 1, '.puml': 4, '.py': 16, '.te': 7, '.txt': 29, '.xml': 29, '.yaml': 1}
  Tree:
  .llm_draft/
    latest/
      OLLAMA_HTTP_LAST.json  (6,184 bytes)
      VHAL_AIDL_RAW_chunk1.txt  (1,889 bytes)
      VHAL_AIDL_RAW_chunk10.txt  (2,016 bytes)
      VHAL_AIDL_RAW_chunk11.txt  (2,193 bytes)
      VHAL_AIDL_RAW_chunk12.txt  (653 bytes)
      VHAL_AIDL_RAW_chunk2.txt  (1,802 bytes)
      VHAL_AIDL_RAW_chunk3.txt  (1,780 bytes)
      VHAL_AIDL_RAW_chunk4.txt  (1,907 bytes)
      VHAL_AIDL_RAW_chunk5.txt  (2

In [33]:
print("=" * 80)
print("MATCHED ANALYSIS RESULTS")
print("=" * 80)
!cat experiments/results/matched_analysis.md

MATCHED ANALYSIS RESULTS
# VHAL — Matched-Agent Fair Comparison

Only agents present in ALL four conditions: **aidl, android_app, android_layout, backend, build, cpp, design_doc, selinux**

Excluded (missing from ≥1 condition): vintf

## 1. Overall Matched Scores

| Condition | Avg Score | Min | Max | Files |
|---|---|---|---|---|
| C1 Baseline | 0.8166 | 0.3479 | 1.0000 | 81 |
| C2 Adaptive | 0.8030 | 0.3479 | 1.0000 | 62 |
| C3 RAG+DSPy | 0.8577 | 0.3792 | 1.0000 | 54 |
| C4 Feedback | 0.8755 | 0.5000 | 1.0000 | 54 |

## 2. Per-Agent Scores (matched only)

| Agent | C1 Baseline | C2 Adaptive | C3 RAG+DSPy | C4 Feedback |
|---|---|---|---|---|
| aidl | 0.8571 (1) | 0.8571 (1) | 0.8408 (7) | 0.8408 (7) |
| android_app | 0.8130 (27) | 0.8147 (17) | 0.9429 (7) | 0.9643 (7) |
| android_layout | 0.9528 (27) | 0.9544 (17) | 0.8714 (7) | 0.8714 (7) |
| backend | 0.5496 (16) | 0.5515 (15) | 0.7564 (16) | 0.8034 (16) |
| build | 0.7750 (1) | 0.8583 (3) | 0.9563 (2) | 0.9563 (2) |
| cpp | 0.955

## 15. Export & save to Drive

In [34]:
from google.colab import files

artifacts = {
    "output_c1":     f"{REPO_DIR}/output",
    "output_c2":     f"{REPO_DIR}/output_adaptive",
    "output_c3":     f"{REPO_DIR}/output_rag_dspy",
    "output_c4":     f"{REPO_DIR}/output_c4_feedback",
    "output_c5":     f"{REPO_DIR}/output_c5",
    "dspy_programs": f"{REPO_DIR}/dspy_opt/saved",
}

for name, src_dir in artifacts.items():
    if Path(src_dir).exists():
        shutil.make_archive(f"/content/{name}", "zip", src_dir)
        shutil.copy(f"/content/{name}.zip", f"{DRIVE_SRC}/{name}.zip")
        print(f"  ✓ {name}.zip ({len(list(Path(src_dir).rglob('*')))} files)")
    else:
        print(f"  ⚠ {name}: not found")

shutil.copytree(f"{REPO_DIR}/experiments/results", f"{DRIVE_SRC}/experiments_results", dirs_exist_ok=True)
print("\n✓ All results saved to Drive")

for name in artifacts:
    zip_path = f"/content/{name}.zip"
    if Path(zip_path).exists():
        files.download(zip_path)

  ✓ output_c1.zip (184 files)
  ✓ output_c2.zip (166 files)
  ✓ output_c3.zip (88 files)
  ✓ output_c4.zip (88 files)
  ✓ dspy_programs.zip (25 files)

✓ All results saved to Drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 16. Run C5 — Advanced Runtime Validation
> Patches FakeVehicleHardware, generates custom VTS tests, and builds HMI app.
> Reads from C4 output + AOSP dump from GCP VM.
> **Prerequisite:** Copy AOSP assets to GCS first (see README Step 8a).

### 16a. Download AOSP assets from GCS

In [ ]:
# Authenticate GCS
from google.colab import auth
auth.authenticate_user()
print("✓ GCS authenticated")

In [ ]:
import subprocess, shutil
from pathlib import Path

GCS_BUCKET = "gs://aosp-thesis-temp"
AOSP_SOURCE = Path(f"{REPO_DIR}/aosp_source")
AOSP_DUMP   = Path(f"{REPO_DIR}/aosp_dump")
AOSP_SOURCE.mkdir(parents=True, exist_ok=True)
AOSP_DUMP.mkdir(parents=True, exist_ok=True)

# Download FakeVehicleHardware.cpp
fake_vhal = AOSP_SOURCE / "FakeVehicleHardware.cpp"
if not fake_vhal.exists():
    r = subprocess.run(["gsutil", "cp", f"{GCS_BUCKET}/FakeVehicleHardware.cpp", str(fake_vhal)],
                       capture_output=True, text=True)
    print(f"  {'✓' if r.returncode == 0 else '❌'} FakeVehicleHardware.cpp")
else:
    print("  ✓ FakeVehicleHardware.cpp already present")

# Download AOSP compiled property ID dump
dump_zip = Path("/content/aosp_dump.zip")
if not dump_zip.exists():
    r = subprocess.run(["gsutil", "cp", f"{GCS_BUCKET}/aosp_dump.zip", str(dump_zip)],
                       capture_output=True, text=True)
    print(f"  {'✓' if r.returncode == 0 else '❌'} aosp_dump.zip downloaded")

# Extract dump
if dump_zip.exists() and not any(AOSP_DUMP.glob("VehicleProperty*.aidl")):
    shutil.unpack_archive(str(dump_zip), "/content/aosp_dump_raw")
    for f in Path("/content/aosp_dump_raw").rglob("VehicleProperty*.aidl"):
        shutil.copy(f, AOSP_DUMP)
    print(f"  ✓ Extracted {len(list(AOSP_DUMP.glob('VehicleProperty*.aidl')))} AIDL dump files")
else:
    print(f"  ✓ AOSP dump: {len(list(AOSP_DUMP.glob('VehicleProperty*.aidl')))} files")

print("\n✓ AOSP assets ready")

### 16b. Run C5 pipeline

In [ ]:
start_ollama()
%cd {REPO_DIR}

# C5 reads: output_c4_feedback/ + aosp_dump/ + aosp_source/FakeVehicleHardware.cpp
# Generates: output_c5/fake_vhal/ + output_c5/vts/ + output_c5/hmi_app/
!python multi_main_c5.py

In [ ]:
# Verify C5 outputs
import os

c5_out = Path(f"{REPO_DIR}/output_c5")
print("=== C5 Output Structure ===")
for subdir in ["fake_vhal", "vts", "hmi_app"]:
    path = c5_out / subdir
    if path.exists():
        files = list(path.rglob("*"))
        print(f"  ✓ {subdir}/: {len(files)} files")
    else:
        print(f"  ⚠ {subdir}/: NOT FOUND")

# Show scores
results_path = c5_out / "c5_results.json"
if results_path.exists():
    import json
    results = json.loads(results_path.read_text())
    print("\n=== C5 Scores ===")
    print(f"  FakeVHAL patch : {results.get('fake_vhal', {}).get('score', 0):.3f}")
    print(f"  VTS tests      : {results.get('vts', {}).get('score', 0):.3f}")
    print(f"  HMI app        : {results.get('hmi_app', {}).get('score', 0):.3f}")
    print(f"  Overall        : {results.get('overall', 0):.3f}")

### 16c. Upload C5 output to GCS (for GCP VM deployment)

In [ ]:
import shutil, subprocess
from pathlib import Path

c5_zip = "/content/output_c5.zip"
c5_dir = f"{REPO_DIR}/output_c5"

if Path(c5_dir).exists():
    shutil.make_archive("/content/output_c5", "zip", c5_dir)
    r = subprocess.run(["gsutil", "cp", c5_zip, f"{GCS_BUCKET}/output_c5.zip"],
                       capture_output=True, text=True)
    print(f"  {'✓' if r.returncode == 0 else '❌'} output_c5.zip → {GCS_BUCKET}")

    # Also save to Drive
    shutil.copy(c5_zip, f"{DRIVE_SRC}/output_c5.zip")
    print(f"  ✓ output_c5.zip → Google Drive")
else:
    print("  ⚠ output_c5/ not found — run C5 pipeline first")

### 16d. GCP VM deployment (run on GCP VM, not Colab)

```bash
# Download C5 output
gsutil cp gs://aosp-thesis-temp/output_c5.zip ~/
unzip ~/output_c5.zip -d ~/

cd ~/aosp-14-auto
source build/envsetup.sh
lunch aosp_cf_x86_64_auto-trunk_staging-userdebug

# Apply FakeVehicleHardware patch
cp ~/output_c5/fake_vhal/FakeVehicleHardware_vss_patch.cpp \
   hardware/interfaces/automotive/vehicle/aidl/impl/fake_impl/hardware/src/FakeVehicleHardware.cpp

# Copy VTS tests
mkdir -p test/vts/vss_vehicle
cp ~/output_c5/vts/* test/vts/vss_vehicle/

# Rebuild (~30 min)
mmm hardware/interfaces/automotive/vehicle/aidl/impl/fake_impl
mmm test/vts/vss_vehicle

# Relaunch Cuttlefish
stop_cvd
launch_cvd --noresume --cpus=4 --memory_mb=4096

# Verify VSS properties are now served
adb -s 0.0.0.0:6520 shell cmd car_service get-property-value 0x1000 0
# Expected: HalPropValue{prop=4096, ...}

# Run VSS VTS tests
atest VtsHalAutomotiveVehicleVss

# Install HMI app
mkdir -p packages/apps/VssDashboard
cp -r ~/output_c5/hmi_app/src ~/output_c5/hmi_app/AndroidManifest.xml \
      ~/output_c5/hmi_app/Android.bp packages/apps/VssDashboard/
mmm packages/apps/VssDashboard
adb -s 0.0.0.0:6520 install -r \
  out/target/product/vsoc_x86_64/system/app/VssDashboardApp/VssDashboardApp.apk

# Verify HMI app
adb -s 0.0.0.0:6520 shell am start -n com.vss.vehicleapp/.MainActivity
adb -s 0.0.0.0:6520 shell cmd car_service inject-vhal-event 0x1000 0 42
adb -s 0.0.0.0:6520 logcat | grep -i "onChangeEvent\|vehicleapp"
```

## 17. Utilities
> Run manually as needed.

In [ ]:
# Restart Ollama
# start_ollama()

In [ ]:
!grep "enum" dspy_opt/validators.py | head -5

In [ ]:
# Force rescore from scratch
!rm experiments/results/*.json
!python rescore_all_conditions.py
!python compare_matched.py

In [ ]:
# Force delete ChromaDB cache (if you need to re-index with new filters)
# import os; os.remove(f"{DRIVE_SRC}/chroma_db.zip"); print("✓ Cache deleted")
# !rm -rf rag/chroma_db

In [ ]:
# Free disk (~300 MB)
# shutil.rmtree(AOSP_SRC_DIR, ignore_errors=True); print("✓ AOSP source removed")